In [ ]:

{
  "narration": "You lunge forward, swinging your mace.",
  "tool_call": {
      "name": "weapon_attack",
      "arguments": {
          "attacker_id": "player",
          "target_id": "goblin",
          "weapon_name": "mace"
      }
  }
}


def dispatch_tool(tool_call, game_state):

    name = tool_call["name"]
    args = tool_call["arguments"]

    if name == "weapon_attack":
        return weapon_attack(game_state, **args)

    elif name == "cast_spell":
        return cast_spell(game_state, **args)

    elif name == "ability_check":
        return ability_check(game_state, **args)

    elif name == "start_combat":
        return start_combat(game_state, **args)

    else:
        return {"error": "Invalid tool"}

In [ ]:
def handle_combat_state(game_state):

    phase = game_state["phase"]

    if phase == "initiative":

        result = roll_initiative(game_state)
        game_state["phase"] = "player_turn"

        narration = ai_gm(
            state_summary(game_state),
            system_result=result
        )
        print("\n" + narration["narration"])


    elif phase == "player_turn":

        player_input = input("\nCombat >> ")

        ai_response = ai_gm(
            state_summary(game_state),
            player_input
        )

        print("\n" + ai_response["narration"])

        if ai_response.get("tool_call"):

            result = dispatch_tool(
                ai_response["tool_call"],
                game_state
            )

            followup = ai_gm(
                state_summary(game_state),
                system_result=result
            )

            print("\n" + followup["narration"])

        if combat_is_over(game_state):
            game_state["phase"] = "combat_end"
        else:
            game_state["phase"] = "enemy_turn"


    elif phase == "enemy_turn":

        # You can either:
        # A) Let LLM choose enemy action via tool call
        # B) Deterministic AI enemy logic

        ai_response = ai_gm(
            state_summary(game_state),
            enemy_turn=True
        )

        if ai_response.get("tool_call"):
            result = dispatch_tool(
                ai_response["tool_call"],
                game_state
            )

            followup = ai_gm(
                state_summary(game_state),
                system_result=result
            )

            print("\n" + followup["narration"])

        if combat_is_over(game_state):
            game_state["phase"] = "combat_end"
        else:
            game_state["phase"] = "player_turn"


    elif phase == "combat_end":

        result = resolve_combat_rewards(game_state)

        narration = ai_gm(
            state_summary(game_state),
            system_result=result
        )

        print("\n" + narration["narration"])

        game_state["mode"] = "exploration"
        game_state["phase"] = None
        game_state["enemy"] = None


def handle_exploration_state(game_state):

    player_input = input("\n>> ")

    ai_response = ai_gm(
        state_summary(game_state),
        player_input
    )

    print("\n" + ai_response["narration"])

    if ai_response.get("tool_call"):

        result = dispatch_tool(
            ai_response["tool_call"],
            game_state
        )

        followup = ai_gm(
            state_summary(game_state),
            system_result=result
        )

        print("\n" + followup["narration"])

In [ ]:
### Set up the starting game state ###

### Step 1 ###
# Create character
john = char.PCFactory().create_basic(name="John", # str
                     race="Halfling", # str name of valid race
                     background="Acolyte", # str name of valid background
                     char_class="Cleric", # str name of valid class
                     ability_method="roll", # one of [standard, roll, point_buy]
                     ability_score_assignment=["INT","CON","STR","DEX","CHA","WIS"], # ["STR","DEX","CON","INT","WIS","CHA"]
                     ability_score_values=None # list of valid point buy numbers [8,10,11,13,15,8]
                     )

### Step 2 ###
# Create the game state
game_state = {
    "mode": "exploration",  # or "combat"
    "phase": None,          # combat phases only
    "player": john,
    "enemy": None,
    "combat": {
        "initiative_order": [],
        "current_turn": None
    },
    "game_over": False
}




while not game_state["game_over"]:

    if game_state["mode"] == "exploration":
        handle_exploration_state(game_state)

    elif game_state["mode"] == "combat":
        handle_combat_state(game_state)